# FreeBindCraft Binder Design — One-Off Sampling Example

![FreeBindCraft Binder Design — One-Off Sampling Example](https://proto-bio.github.io/proto-assets/images/tool/freebindcraft/hero.png)

Design a single protein binder against a target with the FreeBindCraft pipeline
(AlphaFold2 hallucination + ProteinMPNN refinement + OpenMM relaxation, PyRosetta-free).

- Paper: [Pacesa et al., Nature 2025](https://www.nature.com/articles/s41586-025-09429-6)
- Upstream repo (pinned commit `d12747d`): https://github.com/cytokineking/FreeBindCraft/tree/d12747dbc907435622559b81891ad73e0a45c2e4

This example uses tiny iteration counts and `max_trajectories=1` for a
~10-30 minute run on an H100. For production runs (100 binders, 24-72h),
leave `FreeBindCraftConfig` at its upstream defaults and set
`number_of_final_designs=100` on the input.

> **Note: long-running tool — design cells not executed here.** Binder design is a stochastic, filter-driven GPU campaign: a real run launches many trajectories and takes hours to days, and even a scaled-down run is too slow and too uncertain to execute inline. The design cells below are shown for reference and left **unexecuted**; the input / config / output API reference is rendered above. To run a campaign, execute the cells on a GPU.

In [ ]:
from proto_tools.tools.binder_design.freebindcraft import (
    FreeBindCraftConfig,
    FreeBindCraftInput,
    run_freebindcraft_design,
)

## Input API Reference

`FreeBindCraftInput` specifies the target and the desired binder properties.

| Field | Type | Default | Description |
|---|---|---|---|
| `target_pdb` | `str` | *(required)* | Target structure (file path or PDB-format string). |
| `target_chain` | `str` | `"A"` | Chain ID(s) of the frozen target (comma-separated for multi-chain). |
| `target_hotspot_residues` | `str \| None` | `None` | Comma-separated 1-indexed target residue positions the binder must contact (e.g. `"1-10,56,78"`). `None` = unrestricted. |
| `binder_lengths` | `tuple[int, int]` | `(65, 150)` | `(min, max)` binder length range. |
| `binder_name` | `str` | `"binder"` | Project identifier used as a prefix in output filenames. |
| `number_of_final_designs` | `int` | `100` | Target accepted-design count. Set to `1` for one-off sampling. |

The designed binder is always emitted as chain `B` — upstream FreeBindCraft hardcodes this and does not expose it as a setting.

## Config API Reference

`FreeBindCraftConfig` exposes 35 of upstream's 61 `settings_advanced` fields as typed settings
plus a `filter_overrides` dict for per-metric threshold tweaks. The other 26 are hardcoded
internals — file-output / cleanup toggles (animations / plots / MPNN FASTAs / trajectory
pickles / zip archives / intermediate-PDB cleanup), the `sample_models` flag, and
preset-constant dials (contact geometry, AF2 recycle counts, MPNN internals, side-chain
template masking) pinned to upstream defaults — because proto-tools never surfaces those
scratch artifacts or varies those internals. The table here lists the settings you'll most
often tune.

| Field | Type | Default | Description |
|---|---|---|---|
| `design_algorithm` | `Literal["2stage", "3stage", "4stage", "greedy", "mcmc"]` | `"4stage"` | Hallucination algorithm. |
| `max_trajectories` | `int \| bool` | `False` | Cap on hallucination trajectories. `False` = unlimited. Set to a small int for one-off sampling. |
| `soft_iterations` | `int` | `75` | Stage-1 (soft) hallucination iteration count. |
| `temporary_iterations` | `int` | `45` | Stage-2 (temporary) hallucination iteration count. |
| `hard_iterations` | `int` | `5` | Stage-3 (hard) hallucination iteration count. |
| `greedy_iterations` | `int` | `15` | Stage-4 (greedy) hallucination iteration count. |
| `num_seqs` | `int` | `20` | Number of MPNN sequences to sample per trajectory. |
| `max_mpnn_sequences` | `int` | `2` | Max MPNN sequences validated (and considered for acceptance) per trajectory. |
| `filter_overrides` | `dict[str, Any]` | `{}` | Per-metric threshold overrides merged on top of `default_filters.json`. |

See the [FreeBindCraft documentation](https://bio-pro.mintlify.app/tools/binder-design/freebindcraft) for the full 35-field user-facing surface (loss weights, MPNN sampling, sequence-template masking, AA constraints, beta-protein optimisation, and acceptance monitoring).

## Output API Reference

### `FreeBindCraftOutput`

| Field | Type | Description |
|---|---|---|
| `designs` | `list[FreeBindCraftDesign]` | Accepted binder designs (length is at most `FreeBindCraftInput.number_of_final_designs`). |
| `n_trajectories_run` | `int` | Total trajectories attempted before stopping (success or hitting `max_trajectories`). |
| `n_designs_accepted` | `int` | Designs that passed all filters (equals `len(designs)`). |

Plus the standard `BaseToolOutput` metadata fields: `tool_id`, `execution_time`, `success`, `errors`.

### `FreeBindCraftDesign`

| Field | Type | Description |
|---|---|---|
| `design_name` | `str` | Unique design identifier emitted by upstream (e.g. `"binder_l60_s12345_mpnn3"`). |
| `binder_sequence` | `str` | Designed binder amino-acid sequence (1-letter codes). |
| `structure` | `Structure` | Relaxed target+binder complex; B-factors are pLDDT on the 0-100 PDB scale. |
| `metrics` | `FreeBindCraftMetrics` | Per-design averaged metrics evaluated by the filter check. |
| `seed` | `int` | Random seed of the trajectory that produced this design. |
| `interface_aas` | `dict[str, int]` | Amino-acid composition at the binder-target interface. |
| `interface_residues` | `list[int]` | 1-indexed binder residue positions at the interface. |

### `FreeBindCraftMetrics`

Mirrors the **PyRosetta-free** `Average_*` columns in upstream's `final_design_stats.csv` (the columns the filter check evaluates against), with snake_case names. Rosetta-only placeholder columns (`dG`, `dG/dSASA`, `Binder_Energy_Score`, `PackStat`, H-bond counts) are **not** surfaced. Primary metric: `avg_iptm`.

| Metric | Type | Range | Unit |
|---|---|---|---|
| `avg_plddt` | `float` | [0.0, 1.0] | fraction |
| `avg_ptm` | `float` | [0.0, 1.0] | fraction |
| `avg_iptm` | `float` | [0.0, 1.0] | fraction |
| `avg_pae` | `float` | ≥0.0 | Å |
| `avg_ipae` | `float` | ≥0.0 | Å |
| `avg_ipsae` | `float` | [0.0, 1.0] | fraction |
| `avg_iplddt` | `float` | [0.0, 1.0] | fraction |
| `avg_ss_plddt` | `float` | [0.0, 1.0] | fraction |
| `avg_binder_plddt` | `float` | [0.0, 1.0] | fraction |
| `avg_binder_ptm` | `float` | [0.0, 1.0] | fraction |
| `avg_binder_pae` | `float` | ≥0.0 | Å |
| `dSASA` | `float` | ≥0.0 | Å² |
| `interface_sasa_pct` | `float` | [0.0, 100.0] | percent |
| `interface_hydrophobicity` | `float` | [0.0, 100.0] | percent |
| `surface_hydrophobicity` | `float` | [0.0, 1.0] | fraction |
| `shape_complementarity` | `float` | [0.0, 1.0] | fraction |
| `n_interface_residues` | `float` | ≥0.0 | count |
| `binder_helix_pct` | `float` | [0.0, 100.0] | percent |
| `binder_betasheet_pct` | `float` | [0.0, 100.0] | percent |
| `binder_loop_pct` | `float` | [0.0, 100.0] | percent |
| `interface_helix_pct` | `float` | [0.0, 100.0] | percent |
| `interface_betasheet_pct` | `float` | [0.0, 100.0] | percent |
| `interface_loop_pct` | `float` | [0.0, 100.0] | percent |
| `hotspot_rmsd` | `float` | ≥0.0 | Å |
| `target_rmsd` | `float` | ≥0.0 | Å |
| `binder_rmsd` | `float` | ≥0.0 | Å |
| `unrelaxed_clashes` | `float` | ≥0.0 | count |
| `relaxed_clashes` | `float` | ≥0.0 | count |

In [ ]:
from pathlib import Path

# Resolve the in-tree renin test target by walking up to the repo root.
_repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
pdb_path = _repo_root / "tests" / "dummy_data" / "renin_af3.pdb"
print(f"Using target PDB: {pdb_path}")

inputs = FreeBindCraftInput(
    target_pdb=str(pdb_path),
    target_chain="A",
    target_hotspot_residues="56",          # 1-indexed residue position(s) on the target
    binder_lengths=(60, 70),               # (min, max)
    binder_name="example",
    number_of_final_designs=1,             # one-off sampling
)

config = FreeBindCraftConfig(
    max_trajectories=1,                    # cap at 1 trajectory
    soft_iterations=10,                    # tiny iteration budget
    temporary_iterations=5,
    hard_iterations=2,
    greedy_iterations=2,
    num_seqs=2,
    max_mpnn_sequences=1,
    seed=42,                               # reproducible
)

result = run_freebindcraft_design(inputs, config)
print(f"Trajectories run: {result.n_trajectories_run}")
print(f"Designs accepted: {result.n_designs_accepted}")

In [ ]:
if result.designs:
    design = result.designs[0]
    print(f"Design: {design.design_name}")
    print(f"Sequence ({len(design.binder_sequence)} aa): {design.binder_sequence}")
    print(f"Seed: {design.seed}")
    print(f"avg pLDDT: {design.metrics['avg_plddt']:.3f}")
    print(f"avg iPTM:  {design.metrics['avg_iptm']:.3f}")
    print(f"avg ipSAE: {design.metrics['avg_ipsae']:.3f}")
    print(f"Shape complementarity: {design.metrics['shape_complementarity']:.3f}")
else:
    print("No designs passed the filter check — try more trajectories or relaxed filters.")

## Tuning runtime

FreeBindCraft is a stochastic, filter-driven pipeline — most trajectories fail the filter check, so the wall-clock time depends on both the number of accepted designs you want and how strict the filters are. Three useful operating points:

- **One-off sampling (~10-30 min on H100)**: this notebook — `number_of_final_designs=1`, `max_trajectories=1`, tiny iteration counts. Good for smoke-testing the pipeline end-to-end on a new target.
- **Quick scan (~2-4h)**: `number_of_final_designs=5`, `max_trajectories=50`, leave iteration counts at upstream defaults. Enough designs to see if the target is tractable at all.
- **Production run (~24-72h)**: `number_of_final_designs=100`, no `max_trajectories` cap, default iteration counts. Matches the upstream paper's per-target budget.

In [ ]:
# Demonstrate `filter_overrides`: tighten Average_pLDDT and Average_i_pTM beyond
# the defaults. This cell only constructs the config (no GPU / no subprocess) so
# the notebook can be executed end-to-end within reasonable time budgets — to
# actually run, pass `strict_inputs` + `strict_config` to `run_freebindcraft_design`.
strict_inputs = FreeBindCraftInput(
    target_pdb=str(pdb_path),
    target_chain="A",
    target_hotspot_residues="56",
    binder_lengths=(60, 70),
    binder_name="strict",
    number_of_final_designs=1,
)
strict_config = FreeBindCraftConfig(
    max_trajectories=1,
    soft_iterations=10, temporary_iterations=5,
    hard_iterations=2, greedy_iterations=2,
    num_seqs=2, max_mpnn_sequences=1,
    seed=42,
    filter_overrides={
        "Average_pLDDT": {"threshold": 0.85, "higher": True},
        "Average_i_pTM": {"threshold": 0.6, "higher": True},
    },
)

print("Strict config:")
print(f"  filter_overrides = {strict_config.filter_overrides}")
# To actually run: strict_result = run_freebindcraft_design(strict_inputs, strict_config)

In [ ]:
if result.designs:
    result.export("my_binder", file_format="pdb")  # writes a directory of PDBs + stats.json